# Inherent Sample Profiling — Audit Karakteristik Bawaan per Dialog

Tahap kedua: alih-alih menanyakan *config mana yang menang*, kita mengaudit **perilaku ke-N sampel** lintas
eksperimen. Pertanyaan inti:

> Apakah dialog yang berskor tinggi di Eksperimen A juga konsisten tinggi di B, C, dst?

Kita memetakan **profil kesulitan** tiap dialog dan mengelompokkannya:
- **always_high** — kebal terhadap perubahan, selalu bagus di semua config.
- **always_low** — selalu gagal di semua config (kandidat "sulit bawaan").
- **swinger** — tinggi di satu eksperimen, rendah di lain (paling sensitif terhadap variabel).
- **neutral** — di tengah / tak menonjol.

Metode:
1. **Korelasi ranking lintas config** (Spearman & Kendall) — mengukur apakah ada sifat bawaan.
2. **ICC(2,1)** — berapa proporsi variasi skor berasal dari perbedaan antar-dialog vs antar-config.
3. **Klasifikasi profil per dialog** — berbasis percentile per config (robust terhadap beda skala).
4. **Daftar kandidat** — always_low & swinger diekspor untuk inspeksi manual.

> Verifikasi: `unieval.json` tiap config adalah **list per-instance** dengan urutan dialog yang **sama** (paired).

## 1. Loader minimal (mandiri)

In [1]:
import json
from pathlib import Path

def load_json(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"Tidak ditemukan: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

## 2. Muat config & skor komposit per dialog

Sama persis dengan setup paired test Anda — supaya kedua notebook membicarakan dialog yang identik.

In [2]:
import numpy as np, pandas as pd

METHOD = "method3"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}
LING = ["coherence", "relevance", "fluency", "consistency"]

def load_per_instance(folder):
    return pd.DataFrame(load_json(f"{folder}/unieval.json"))

per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
print("Jumlah dialog per config:", {k: len(v) for k, v in per.items()}, "-> pakai", n, "(paired)")
for k in per:
    per[k] = per[k].iloc[:n].reset_index(drop=True)
    per[k]["_ling"] = per[k][LING].mean(axis=1)

# matriks skor komposit: baris = dialog, kolom = config
S = pd.DataFrame({k: per[k]["_ling"].values for k in per})
S.index.name = "dialog_id"
print("\nMatriks skor komposit (head):")
print(S.head().round(4).to_string())

Jumlah dialog per config: {'margin_part': 499, 'margin_full': 499, 'finetune': 499} -> pakai 499 (paired)

Matriks skor komposit (head):
           margin_part  margin_full  finetune
dialog_id                                    
0               0.9549       0.9607    0.9567
1               0.9072       0.9525    0.8925
2               0.8701       0.9569    0.9467
3               0.9395       0.9068    0.9373
4               0.7771       0.8938    0.9700


## 3. Apakah ada sifat bawaan? — Korelasi ranking lintas config

Jika dialog sulit cenderung sulit di mana-mana, ranking skor antar config akan **berkorelasi tinggi**.
Spearman/Kendall dipilih karena hanya peduli urutan (bukan skala absolut), cocok untuk membandingkan
"kesulitan relatif" lintas eksperimen.

In [3]:
from scipy.stats import spearmanr, kendalltau
from itertools import combinations

print("="*64)
print("KORELASI RANKING ANTAR CONFIG (skor komposit _ling)")
print("="*64)
print("Spearman ρ tinggi (mendekati 1) -> dialog sulit/mudah konsisten lintas config (ADA sifat bawaan).\n")

rho_mat = pd.DataFrame(np.eye(S.shape[1]), index=S.columns, columns=S.columns)
for a, b in combinations(S.columns, 2):
    rho, p_rho = spearmanr(S[a], S[b])
    tau, p_tau = kendalltau(S[a], S[b])
    rho_mat.loc[a, b] = rho_mat.loc[b, a] = rho
    print(f"  {a:12s} vs {b:12s}  Spearman ρ={rho:+.3f} (p={p_rho:.2g})   Kendall τ={tau:+.3f} (p={p_tau:.2g})")

print("\nMatriks Spearman ρ:")
print(rho_mat.round(3).to_string())
mean_rho = rho_mat.values[np.triu_indices_from(rho_mat.values, k=1)].mean()
print(f"\nRata-rata ρ antar-pasangan = {mean_rho:+.3f}")
print("  ρ>~0.7  : sifat bawaan kuat (kesulitan dialog dominan, config sekadar menggeser).")
print("  ρ<~0.3  : sifat bawaan lemah (config benar-benar mengubah dialog mana yang mudah/sulit).")

KORELASI RANKING ANTAR CONFIG (skor komposit _ling)
Spearman ρ tinggi (mendekati 1) -> dialog sulit/mudah konsisten lintas config (ADA sifat bawaan).

  margin_part  vs margin_full   Spearman ρ=+0.368 (p=1.8e-17)   Kendall τ=+0.254 (p=2e-17)
  margin_part  vs finetune      Spearman ρ=+0.379 (p=1.8e-18)   Kendall τ=+0.262 (p=2.4e-18)
  margin_full  vs finetune      Spearman ρ=+0.322 (p=1.6e-13)   Kendall τ=+0.223 (p=1.1e-13)

Matriks Spearman ρ:
             margin_part  margin_full  finetune
margin_part        1.000        0.368     0.379
margin_full        0.368        1.000     0.322
finetune           0.379        0.322     1.000

Rata-rata ρ antar-pasangan = +0.356
  ρ>~0.7  : sifat bawaan kuat (kesulitan dialog dominan, config sekadar menggeser).
  ρ<~0.3  : sifat bawaan lemah (config benar-benar mengubah dialog mana yang mudah/sulit).


## 4. ICC(2,1) — berapa porsi variasi yang "bawaan dialog"

Intraclass Correlation menjawab: dari total variasi skor, berapa yang disebabkan **perbedaan antar dialog**
(sifat bawaan) versus perbedaan antar config + error. ICC mendekati 1 = skor sangat ditentukan oleh dialognya,
config hampir tak berpengaruh.

In [4]:
def icc_2_1(M):
    """ICC(2,1) two-way random, single rater. M: baris=subjek(dialog), kolom=rater(config)."""
    M = np.asarray(M, float)
    n_s, k = M.shape
    grand = M.mean()
    row_m = M.mean(axis=1)
    col_m = M.mean(axis=0)
    SS_total = ((M - grand) ** 2).sum()
    SS_row   = k * ((row_m - grand) ** 2).sum()
    SS_col   = n_s * ((col_m - grand) ** 2).sum()
    SS_err   = SS_total - SS_row - SS_col
    MS_row = SS_row / (n_s - 1)
    MS_col = SS_col / (k - 1)
    MS_err = SS_err / ((n_s - 1) * (k - 1))
    icc = (MS_row - MS_err) / (MS_row + (k - 1) * MS_err + k * (MS_col - MS_err) / n_s)
    return icc

icc = icc_2_1(S.values)
print("="*64)
print(f"ICC(2,1) = {icc:.3f}")
print("="*64)
label = ("buruk (<0.5): config sangat mengubah skor" if icc < 0.5 else
         "sedang (0.5-0.75)" if icc < 0.75 else
         "baik (0.75-0.9): sifat bawaan dialog dominan" if icc < 0.9 else
         "sangat baik (>0.9): skor hampir sepenuhnya ditentukan dialog")
print("Interpretasi:", label)

ICC(2,1) = 0.196
Interpretasi: buruk (<0.5): config sangat mengubah skor


## 5. Klasifikasi profil tiap dialog

Tiap dialog diubah ke **percentile dalam config-nya sendiri** (0=terburuk, 1=terbaik di config itu) agar adil
lintas config yang skala absolutnya beda. Lalu:
- **always_high**: percentile minimal (di semua config) ≥ `HI` → kebal, selalu unggul.
- **always_low**: percentile maksimal ≤ `LO` → selalu gagal di mana-mana.
- **swinger**: rentang percentile (max−min) ≥ `SWING` → tinggi di satu eksperimen, rendah di lain.
- selebihnya **neutral**.

Ambang bisa Anda setel.

In [5]:
HI, LO, SWING = 0.80, 0.10, 0.50  # ambang percentile

# percentile per config (rank relatif dalam kolom)
P = S.rank(pct=True)
P.columns = [c + "_pct" for c in P.columns]

prof = pd.DataFrame(index=S.index)
prof = prof.join(S.round(4)).join(P.round(3))
prof["pct_min"]   = P.min(axis=1).round(3)
prof["pct_max"]   = P.max(axis=1).round(3)
prof["pct_range"] = (P.max(axis=1) - P.min(axis=1)).round(3)
prof["mean_score"] = S.mean(axis=1).round(4)
prof["std_score"]  = S.std(axis=1, ddof=1).round(4)

def classify(r):
    if r["pct_range"] >= SWING:     return "swinger"
    if r["pct_min"]   >= HI:        return "always_high"
    if r["pct_max"]   <= LO:        return "always_low"
    return "neutral"

prof["profile"] = prof.apply(classify, axis=1)

print("="*64)
print("DISTRIBUSI PROFIL")
print("="*64)
print(prof["profile"].value_counts().to_string())
print(f"\nTotal dialog: {len(prof)}  | ambang HI={HI} LO={LO} SWING={SWING}")

DISTRIBUSI PROFIL
profile
neutral        322
swinger        151
always_high     23
always_low       3

Total dialog: 499  | ambang HI=0.8 LO=0.1 SWING=0.5


## 6. Daftar kandidat: always_low & swinger

Ini yang Anda cari: sampel yang **selalu rendah** (sulit bawaan) dan sampel **swinger** (rendah di satu
eksperimen, tinggi di lain). Diurutkan agar yang paling ekstrem muncul di atas.

In [6]:
score_cols = list(S.columns)
show_cols = score_cols + ["mean_score", "pct_min", "pct_max", "pct_range", "profile"]

always_low = prof[prof["profile"] == "always_low"].sort_values("mean_score")
swingers   = prof[prof["profile"] == "swinger"].sort_values("pct_range", ascending=False)
always_high = prof[prof["profile"] == "always_high"].sort_values("mean_score", ascending=False)

print("="*64)
print(f"ALWAYS_LOW — sulit bawaan, selalu gagal ({len(always_low)} dialog)")
print("="*64)
print(always_low[show_cols].head(20).to_string())

print("\n" + "="*64)
print(f"SWINGER — sensitif: rendah di satu config, tinggi di lain ({len(swingers)} dialog)")
print("="*64)
print(swingers[show_cols].head(20).to_string())

print("\n" + "="*64)
print(f"ALWAYS_HIGH — kebal, selalu unggul ({len(always_high)} dialog)")
print("="*64)
print(always_high[show_cols].head(10).to_string())

ALWAYS_LOW — sulit bawaan, selalu gagal (3 dialog)
           margin_part  margin_full  finetune  mean_score  pct_min  pct_max  pct_range     profile
dialog_id                                                                                         
380             0.6096       0.6644    0.5861      0.6200    0.022    0.064      0.042  always_low
325             0.6531       0.7250    0.5099      0.6293    0.042    0.090      0.048  always_low
59              0.6099       0.6726    0.6569      0.6465    0.024    0.096      0.072  always_low

SWINGER — sensitif: rendah di satu config, tinggi di lain (151 dialog)
           margin_part  margin_full  finetune  mean_score  pct_min  pct_max  pct_range  profile
dialog_id                                                                                      
214             0.9658       0.9236    0.4788      0.7894    0.030    0.982      0.952  swinger
292             0.5157       0.9617    0.8763      0.7846    0.014    0.962      0.948  swinge

### 6b. Untuk tiap swinger: di config mana tinggi, di config mana rendah

Memudahkan menjawab "rendah di Eksperimen A tapi tinggi di Eksperimen B" secara eksplisit.

In [7]:
if len(swingers):
    detail = pd.DataFrame(index=swingers.index)
    detail["best_config"]  = S.loc[swingers.index].idxmax(axis=1)
    detail["worst_config"] = S.loc[swingers.index].idxmin(axis=1)
    detail["best_score"]   = S.loc[swingers.index].max(axis=1).round(4)
    detail["worst_score"]  = S.loc[swingers.index].min(axis=1).round(4)
    detail["gap"]          = (detail["best_score"] - detail["worst_score"]).round(4)
    detail["pct_range"]    = swingers["pct_range"]
    detail = detail.sort_values("gap", ascending=False)
    print("SWINGER — detail arah perubahan (urut gap terbesar):")
    print(detail.head(20).to_string())
else:
    print("Tidak ada swinger pada ambang ini. Turunkan SWING bila ingin lebih sensitif.")

SWINGER — detail arah perubahan (urut gap terbesar):
           best_config worst_config  best_score  worst_score     gap  pct_range
dialog_id                                                                      
220        margin_part     finetune      0.9575       0.2439  0.7136      0.904
202        margin_part  margin_full      0.9505       0.3894  0.5611      0.814
303        margin_full     finetune      0.9507       0.3928  0.5579      0.822
360           finetune  margin_part      0.9544       0.3985  0.5559      0.886
475           finetune  margin_full      0.9478       0.4037  0.5441      0.826
55         margin_full     finetune      0.9342       0.4014  0.5328      0.625
397           finetune  margin_part      0.9328       0.4034  0.5294      0.663
192        margin_part     finetune      0.9310       0.4040  0.5270      0.589
138        margin_full     finetune      0.9606       0.4399  0.5207      0.926
323        margin_part  margin_full      0.9525       0.4414  0.511

## 7. (Opsional) Profil per metrik, bukan hanya komposit

Kadang dialog selalu rendah hanya di satu metrik (mis. `fluency`) tapi normal di lainnya. Sel ini
menghitung, per dialog & per metrik, rata-rata lintas config + apakah dialog itu konsisten di kuartil bawah.

In [8]:
metric_low = pd.DataFrame(index=range(n))
for m in LING:
    Mm = pd.DataFrame({k: per[k][m].values for k in per})
    Pm = Mm.rank(pct=True)
    metric_low[m + "_mean"]    = Mm.mean(axis=1).round(4)
    metric_low[m + "_pct_max"] = Pm.max(axis=1).round(3)  # ≤ LO di semua config -> selalu rendah pada metrik ini
    metric_low[m + "_alwaysLow"] = (Pm.max(axis=1) <= LO)

always_low_cols = [m + "_alwaysLow" for m in LING]
metric_low["n_metrics_alwaysLow"] = metric_low[always_low_cols].sum(axis=1)

print("Dialog dengan ≥1 metrik 'selalu rendah' lintas config:")
sel = metric_low[metric_low["n_metrics_alwaysLow"] >= 1].sort_values("n_metrics_alwaysLow", ascending=False)
print(sel.head(20).to_string())
print(f"\nJumlah dialog dengan minimal satu metrik selalu rendah: {len(sel)}")

Dialog dengan ≥1 metrik 'selalu rendah' lintas config:
     coherence_mean  coherence_pct_max  coherence_alwaysLow  relevance_mean  relevance_pct_max  relevance_alwaysLow  fluency_mean  fluency_pct_max  fluency_alwaysLow  consistency_mean  consistency_pct_max  consistency_alwaysLow  n_metrics_alwaysLow
325          0.5614              0.098                 True          0.4580              0.066                 True        0.7706            0.631              False            0.7273                0.140                  False                    2
395          0.6262              0.088                 True          0.6872              0.162                False        0.6938            0.339              False            0.7075                0.070                   True                    2
39           0.4427              0.074                 True          0.4476              0.122                False        0.7577            0.818              False            0.7040               

## 8. Ekspor

In [9]:
prof.to_csv("inherent_profile_2000d10ep.csv")
always_low[show_cols].to_csv("candidates_always_low.csv")
swingers[show_cols].to_csv("candidates_swinger.csv")
rho_mat.to_csv("rank_correlation_matrix.csv")
print("Disimpan:")
print("  inherent_profile_2000d10ep.csv  (profil semua dialog)")
print("  candidates_always_low.csv       (sulit bawaan)")
print("  candidates_swinger.csv          (sensitif / swinger)")
print("  rank_correlation_matrix.csv     (matriks Spearman)")
print(f"\nRingkas: ICC={icc:.3f}, rata-rata Spearman ρ={mean_rho:+.3f}")
print("""
CARA BACA KESELURUHAN:
- ρ tinggi + ICC tinggi  -> kesulitan dialog adalah sifat BAWAAN; config hanya menggeser sedikit.
  Maka 'always_low' benar-benar sampel sulit universal yang layak diinspeksi.
- ρ rendah / banyak swinger -> dialog mana yang sulit sangat tergantung config; perubahan eksperimen
  bekerja dengan menukar untung-rugi antar dialog, bukan menaikkan semua.
- Pakai candidates_always_low untuk error analysis (kenapa model selalu gagal di sampel ini),
  dan candidates_swinger untuk menunjukkan kasus di mana satu config jelas menolong/merusak.""")

Disimpan:
  inherent_profile_2000d10ep.csv  (profil semua dialog)
  candidates_always_low.csv       (sulit bawaan)
  candidates_swinger.csv          (sensitif / swinger)
  rank_correlation_matrix.csv     (matriks Spearman)

Ringkas: ICC=0.196, rata-rata Spearman ρ=+0.356

CARA BACA KESELURUHAN:
- ρ tinggi + ICC tinggi  -> kesulitan dialog adalah sifat BAWAAN; config hanya menggeser sedikit.
  Maka 'always_low' benar-benar sampel sulit universal yang layak diinspeksi.
- ρ rendah / banyak swinger -> dialog mana yang sulit sangat tergantung config; perubahan eksperimen
  bekerja dengan menukar untung-rugi antar dialog, bukan menaikkan semua.
- Pakai candidates_always_low untuk error analysis (kenapa model selalu gagal di sampel ini),
  dan candidates_swinger untuk menunjukkan kasus di mana satu config jelas menolong/merusak.
